# 3 - 5 Candle Drop and Pop

![alt text](https://i.imgur.com/BD7CeaI.png)

In [ ]:
import os 
import plotly.graph_objects as go
os.chdir("c:\\Users\\vynde\\Documents\\GitHub\\Algotrader")

In [ ]:
def streak_count(cond):
    """Count the number of consecutive True values"""
    group_id = cond.astype(int).diff().fillna(0).eq(-1).cumsum()
    # .astype(int) -> convert True/False to 1/0
    # .diff() -> find changes {-1: True->False, 1: False->True, 0: no change}
    # .fillna(0) -> first value
    # .eq(-1) -> find where the change is -1 (True->False)
    # .cumsum() -> results in a 'group id' for each consecutive streak
    return cond.groupby(group_id).transform("cumsum")
    # .groupby(group_id) -> group by the group id

def consecutive_returns(returns):
    
    # group id for consecutive returns
    # new id every time return changes between >0 <0 or 0
    values = returns.copy()
    values[values == 0] = 0
    values[values > 0] = 1
    values[values < 0] = -1
    cond = values.diff()
    cond[cond != 0] = 1
    group_id = cond.cumsum()
    return returns.groupby(group_id).transform("cumsum")


In [ ]:
df = pd.read_csv("EURUSD_2024_1min.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df.set_index("datetime", inplace=True)
df["up"] = df["close"] > df["open"]  # green candle
df["down"] = df["close"] < df["open"]  # red candle

df["lower_highs"] = streak_count(df.high < df.high.shift(1))
df["lower_lows"] = streak_count(df.low < df.low.shift(1))
df["higher_highs"] = streak_count(df.high > df.high.shift(1))
df["higher_lows"] = streak_count(df.low > df.low.shift(1))
df["upcount"] = streak_count(df.up)
df["downcount"] = streak_count(df.down)

df["return"] = df["close"].diff().round(5)
df["consec_returns"] = consecutive_returns(df["return"].round(5)).round(5)

stage_names = {
    0: "idle",
    1: "strong_advance",  # min 3 green candles
    2: "3-5 candle decline",  # min 3 red candles / 60% max retracement
    3: "reversal"}  # green candle

df["stage"] = 0
stages = []
upmoves = []
downmoves = []
for i in range(len(df)):
    if i == 0:
        stage = 0
        upstreak = 0
        downstreak = 0

    upstreak = upstreak + 1 if df["up"].iloc[i] else 0
    downstreak = downstreak + 1 if df["down"].iloc[i] else 0

    if stage == 0:
        # init
        upmove = None
        downmove = None
        if upstreak >= 3:
            # advance stage
            stage = 1

    elif stage == 1:  # 3 green candles
        # save the upmove after first red candle
        if df["down"].iloc[i]:
            upmove = upmove if upmove else df["consec_returns"].iloc[i-1]
        # advance stage
        if downstreak >= 3:
            stage = 2

    elif stage == 2:  # 3 red candles
        if df["up"].iloc[i]:
            # get downmove after first green candle
            downmove = df["consec_returns"].abs().iloc[i-1]
            # advance stage and save values
            if (downmove <= 0.6 * upmove) and (downmove > 0):
                upmoves.append(upmove)
                downmoves.append(downmove)
                stage = 3
            # reset stage
            else:
                stage = 0
    
    # reset stage after final stage
    elif stage == 3:
        stage = 0

    stages.append(stage)
df["stage"] = stages

entries = (df[df.stage == 3])

# This was the vectorized approach. Too complicated to get it working
#numred = 4
#entries = df[
#    # first green candle
#    (df.upcount==1) &
#    # 3 red candles before with lower high and lower low
#    df.shift(1).eval(f"downcount=={numred}") &# and lh>={numred} and ll>={numred}") &
#    # max 60% retracement from last green candles
#    (df["consec_returns"].shift(1).abs() < 0.6 * df["consec_returns"].shift(numred+1).abs()) &
#    # 3 green candles 
#    df.shift(numred+1).eval(f"upcount>={numred}")
#    ]
#
print(len(entries))

for i in range(len(entries)):
    start = entries.index[i] - pd.Timedelta(minutes=20)
    end = entries.index[i] + pd.Timedelta(minutes=20)
    break
df.loc[start:end][["close", "return", "consec_returns", "stage"]]


In [ ]:
# plot the first 3 reversals. for each reversal plot 20 candles before and 20 after the reversal
for i in range(3):
    start = entries.index[i] - pd.Timedelta(minutes=20)
    end = entries.index[i] + pd.Timedelta(minutes=20)
    fig = go.Figure()
    text = [f"CReturns: {round(r*1e5)}" for r in df.loc[start:end]["consec_returns"]]

    fig.add_trace(go.Candlestick(x=df.loc[start:end].index,
                                 open=df.loc[start:end]["open"],
                                 high=df.loc[start:end]["high"],
                                 low=df.loc[start:end]["low"],
                                 close=df.loc[start:end]["close"],
                                 customdata=df[["consec_returns"]].values,
                                 text=text
))
    fig.update_layout(title=f"Reversal {entries.index[i]} / upmove {upmoves[i]} downmove {downmoves[i]} / Retracement {downmoves[i]/upmoves[i]}", xaxis_title="Time", yaxis_title="Price")
    # annotate candles with the stage
    for j in range(len(df.loc[start:end])):
        fig.add_annotation(x=df.loc[start:end].index[j],
                            y=df.loc[start:end]["high"].iloc[j],
                            text=f"{df.loc[start:end].iloc[j]['stage']}",
                            ax=0,
                            ay=-10,)
    fig.show()
